# Corbel YOLOv8 Transfer Learning: yolov8n → yolov8s

Train yolov8s using pretrained yolov8n backbone (from train2.onnx, 142 epochs).

**Strategy**: Transfer Learning (warm start)
- Load trained yolov8n backbone (saves 142 epochs of learning)
- Initialize yolov8s with injected backbone weights
- Fine-tune for 80-100 epochs at 10x lower learning rate
- **Expected**: 3x faster than training from scratch + better accuracy

**Augmentation tuning** (from corbel/context.md investigation):
- copy_paste: 0.3 (small-object recall)
- mixup: 0.1 (mild regularization)
- degrees: 5.0 (rotation for scanned plans)
- erasing: 0.25 (preserve thin walls)

## Cell 1: GPU Check & Mount Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Dataset Setup

In [ ]:
!unzip -q /content/drive/MyDrive/fyp-dataset/corbel-dataset.zip -d /content/
!sed -i 's|^path:.*|path: /content/dataset|' /content/dataset/cubicasa.yaml
print("✓ Dataset ready at /content/dataset")
!head -10 /content/dataset/cubicasa.yaml

## Cell 3: Install Ultralytics

In [ ]:
!pip install -q ultralytics
print("✓ Ultralytics installed")

## Cell 4: Transfer Learning - yolov8n → yolov8s

In [ ]:
from ultralytics import YOLO
import os

# Step 1: Load pretrained yolov8n from train2 checkpoint (142 epochs already trained)
print("Step 1: Loading pretrained yolov8n backbone (train2, 142 epochs)...")
pretrained_path = '/content/drive/MyDrive/corbel-runs/corbel/weights/best.pt'

if not os.path.exists(pretrained_path):
    print(f"ERROR: Pretrained checkpoint not found at {pretrained_path}")
    print("Make sure train2.onnx was saved to corbel-runs/corbel/weights/best.pt")
else:
    pretrained_model = YOLO(pretrained_path)
    print(f"✓ Loaded pretrained yolov8n from {pretrained_path}")

    # Step 2: Initialize yolov8s architecture
    print("\nStep 2: Initializing yolov8s architecture...")
    model = YOLO('yolov8s.pt')
    print("✓ yolov8s initialized")

    # Step 3: Transfer backbone weights
    print("\nStep 3: Transferring backbone weights (yolov8n → yolov8s)...")
    model.model.backbone.load_state_dict(
        pretrained_model.model.backbone.state_dict()
    )
    print("✓ Backbone weights transferred")

    # Step 4: Fine-tune with warm learning rate
    print("\nStep 4: Fine-tuning yolov8s (100 epochs, lr0=0.001 warm start)...\n")
    model.train(
        data='/content/dataset/cubicasa.yaml',
        epochs=100,           # Warm start: fewer epochs needed
        patience=40,          # Early stopping
        imgsz=640,
        save_period=1,        # Save checkpoint every epoch
        optimizer='AdamW',
        batch=-1,             # Auto batch size
        lr0=0.001,            # Warm start: 10x lower learning rate
        lrf=0.0001,
        warmup_epochs=5,      # Longer warmup for stable convergence
        # Augmentation tuning (from corbel/context.md investigation):
        copy_paste=0.3,       # Small-object (door/window) recall
        mixup=0.1,            # Mild, preserve line geometry
        degrees=5.0,          # Rotation for hand-scanned plans
        erasing=0.25,         # Preserve thin walls
        # Output
        project='/content/drive/MyDrive/corbel-runs',
        name='corbel-runA-transfer',
        exist_ok=True,
    )
    print("\n✓ Training complete!")

## Cell 5: Export ONNX

In [ ]:
from ultralytics import YOLO
import os

best_path = '/content/drive/MyDrive/corbel-runs/corbel-runA-transfer/weights/best.pt'

if os.path.exists(best_path):
    print(f"Exporting ONNX from {best_path}...\n")
    model = YOLO(best_path)
    onnx_path = model.export(format='onnx', imgsz=640, opset=17)
    print(f"\n✓ ONNX exported: {onnx_path}")
    print(f"\nNext steps:")
    print(f"1. Download the ONNX from Drive to local: ml/train3.onnx")
    print(f"2. Benchmark: node --experimental-strip-types corbel/scripts/benchmark-local-yolo.ts --split test --n 400 --model ml/train3.onnx")
    print(f"3. Compare against baseline (80.1% overall F1) and decide on Run B")
else:
    print(f"ERROR: best.pt not found at {best_path}")
    print("Make sure training completed successfully before exporting.")